In [ ]:
%pip install mitreattack-python


In [22]:
# !wget -q -O "cisa_AA24-326A.pdf" https://www.cisa.gov/sites/default/files/2024-11/aa24-326a-enhancing-cyber-resilience-insights-from-cisa-red-team-assessment_0.pdf
# !curl -s -L -o "enterprise_attack.json" https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json

In [23]:
from mitreattack.stix20 import MitreAttackData

attacks = MitreAttackData("enterprise_attack.json")

technique_to_mitigations = attacks.get_all_mitigations_mitigating_all_techniques()
technique_to_software = attacks.get_all_software_using_all_techniques()

techniques = attacks.get_techniques(
    include_subtechniques=True,
    remove_revoked_deprecated=True,
)

print(f"Total techniques: {len(techniques)}")

Total techniques: 691


In [24]:
attck_chunks = []

for technique in techniques:
    # Clean API calls replace all manual external_references loops
    mitre_id = attacks.get_attack_id(technique.id) or ""
    tactics  = [t.name for t in attacks.get_tactics_by_technique(technique.id)]

    # Software (tools + malware) using this technique
    software_entries = technique_to_software.get(technique.id, [])
    software_names   = [entry["object"].name for entry in software_entries]

    # Mitigations for this technique
    mitigation_entries = technique_to_mitigations.get(technique.id, [])
    mitigation_names   = [entry["object"].name for entry in mitigation_entries]

    # Separate tools from malware for richer metadata
    tools   = [e["object"].name for e in software_entries 
                if e["object"].type == "tool"]
    malware = [e["object"].name for e in software_entries 
                if e["object"].type == "malware"]

    is_subtechnique = "." in mitre_id

    chunk_text = f"""MITRE ATT&CK Technique: {technique.name}
ID: {mitre_id}
Type: {"Sub-technique" if is_subtechnique else "Technique"}
Tactics: {", ".join(tactics)}

Description:
{getattr(technique, "description", "Not available").strip()}

Detection:
{getattr(technique, "x_mitre_detection", "Not documented").strip()}

Associated Tools: {", ".join(tools) if tools else "None documented"}
Associated Malware: {", ".join(malware) if malware else "None documented"}
Mitigations: {", ".join(mitigation_names) if mitigation_names else "None documented"}""".strip()

    attck_chunks.append({
        "text": chunk_text,
        "metadata": {
            "source":          "MITRE ATT&CK",
            "id":              mitre_id,
            "name":            technique.name,
            "tactics":         tactics,
            "is_subtechnique": is_subtechnique,
            "tools":           tools,
            "malware":         malware,
            "mitigations":     mitigation_names,
        }
    })

print(f"ATT&CK chunks built: {len(attck_chunks)}")

# Sanity check
sample = attck_chunks[0]
print(f"\nSample chunk — {sample['metadata']['id']}: {sample['metadata']['name']}")
print(sample["text"][:500])

ATT&CK chunks built: 691

Sample chunk — T1055.011: Extra Window Memory Injection
MITRE ATT&CK Technique: Extra Window Memory Injection
ID: T1055.011
Type: Sub-technique
Tactics: Privilege Escalation, Defense Evasion

Description:
Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or reg


In [29]:
import pymupdf 
from langchain_text_splitters import RecursiveCharacterTextSplitter

def parse_and_chunk_pdf(path, source_name, chunk_size=800, overlap=150):
    doc = pymupdf.open(path)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    
    chunks = []
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        if not text:
            continue
        for j, split in enumerate(splitter.split_text(text)):
            if len(split.strip()) < 50:
                continue
            chunks.append({
                "text": split.strip(),
                "metadata": {
                    "source": source_name,
                    "page": i + 1,
                    "chunk_index": j,
                }
            })
    return chunks

cisa_chunks = parse_and_chunk_pdf("cisa_AA24-326A.pdf", "CISA Red Team Assessment")
print(f"CISA chunks built: {len(cisa_chunks)}")

CISA chunks built: 135


In [30]:
PTES_SECTIONS = {
    "Pre-engagement Interactions": "http://www.pentest-standard.org/index.php/Pre-engagement",
    "Intelligence Gathering":      "http://www.pentest-standard.org/index.php/Intelligence_Gathering",
    "Threat Modeling":             "http://www.pentest-standard.org/index.php/Threat_Modeling",
    "Vulnerability Analysis":      "http://www.pentest-standard.org/index.php/Vulnerability_Analysis",
    "Exploitation":                "http://www.pentest-standard.org/index.php/Exploitation",
    "Post Exploitation":           "http://www.pentest-standard.org/index.php/Post_Exploitation",
    "Reporting":                   "http://www.pentest-standard.org/index.php/Reporting",
}

In [33]:
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

def scrape_ptes_section(url, section_name):
    """Scrape a PTES section and return clean text."""
    headers = {"User-Agent": "Mozilla/5.0"}
    
    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
    except Exception as e:
        print(f"Failed to fetch {section_name}: {e}")
        return None

    soup = BeautifulSoup(r.text, "html.parser")

    # PTES uses MediaWiki - content is in div#mw-content-text
    content_div = soup.find("div", {"id": "mw-content-text"})
    if not content_div:
        # Fallback to body text
        content_div = soup.find("body")

    if not content_div:
        print(f"No content found for {section_name}")
        return None

    # Remove navigation, edit links, and other noise
    for tag in content_div.find_all(["script", "style", "table", 
                                      "sup", "span.editsection"]):
        tag.decompose()

    # Extract clean text preserving paragraph structure
    text = content_div.get_text(separator="\n")
    
    # Clean up excessive whitespace
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]  # remove empty lines
    clean_text = "\n\n".join(lines)
    
    return clean_text


def chunk_ptes_section(text, section_name, chunk_size=800, overlap=150):
    """Chunk a PTES section into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    
    chunks = []
    for i, split in enumerate(splitter.split_text(text)):
        if len(split.strip()) < 50:
            continue
        chunks.append({
            "text": split.strip(),
            "metadata": {
                "source": "PTES",
                "section": section_name,
                "chunk_index": i,
            }
        })
    return chunks

ptes_chunks = []

for section_name, url in PTES_SECTIONS.items():
    print(f"Scraping: {section_name}...")
    text = scrape_ptes_section(url, section_name)
    
    if text:
        chunks = chunk_ptes_section(text, section_name)
        ptes_chunks.extend(chunks)
        print(f"  → {len(chunks)} chunks")
    else:
        print(f"  → FAILED")

print(f"\nTotal PTES chunks: {len(ptes_chunks)}")

Scraping: Pre-engagement Interactions...
  → 72 chunks
Scraping: Intelligence Gathering...
  → 75 chunks
Scraping: Threat Modeling...
  → 35 chunks
Scraping: Vulnerability Analysis...
  → 49 chunks
Scraping: Exploitation...
  → 30 chunks
Scraping: Post Exploitation...
  → 56 chunks
Scraping: Reporting...
  → 22 chunks

Total PTES chunks: 339


In [34]:
# Check a sample from each section
seen_sections = set()
for chunk in ptes_chunks:
    section = chunk["metadata"]["section"]
    if section not in seen_sections:
        print(f"\n--- {section} ---")
        print(chunk["text"][:300])
        seen_sections.add(section)


--- Pre-engagement Interactions ---
Contents

1

Overview

2

Introduction to Scope

3

Metrics for Time Estimation

4

Scoping Meeting

5

Additional Support Based on Hourly Rate

6

Questionnaires

7

General Questions

7.1

Network Penetration Test

7.2

Web Application Penetration Test

7.3

Wireless Network Penetration Test

7.4


--- Intelligence Gathering ---
Contents

1

General

1.1

Background Concepts

1.1.1

Level 1 Information Gathering

1.1.2

Level 2 Information Gathering

1.1.3

Level 3 Information Gathering

2

Intelligence Gathering

2.1

What it is

2.2

Why do it

2.3

What is it not

3

Target Selection

3.1

Identification and Naming of Ta

--- Threat Modeling ---
Contents

1

General

1.1

High level threat modeling process

1.2

Example

1.3

High level modeling tools

2

Business Asset Analysis

2.1

Organizational Data

2.1.1

Policies, Plans, and Procedures

2.1.2

Product Information (e.g. trade secrets, R&D data)

2.1.3

Marketing Information (plans, r

--

In [ ]:
all_chunks = attck_chunks + cisa_chunks + ptes_chunks

from collections import Counter
sources = [c["metadata"]["source"] for c in all_chunks]
print("Chunks per source:")
for source, count in Counter(sources).items():
    print(f"  {source}: {count}")
    

Chunks per source:
  MITRE ATT&CK: 691
  CISA Red Team Assessment: 135
  PTES: 339


In [36]:
from FlagEmbedding import BGEM3FlagModel

embedder = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print("BGE-M3 loaded")

ModuleNotFoundError: No module named 'FlagEmbedding'

In [ ]:
def truncate_chunk(text, max_chars=3000):
    """Safety truncation before embedding."""
    return text[:max_chars] if len(text) > max_chars else text

all_texts = [truncate_chunk(c["text"]) for c in all_chunks]

print(f"Embedding {len(all_texts)} chunks...")

encoded = embedder.encode(
    all_texts,
    return_dense=True,
    return_sparse=True,
    batch_size=16,          # reduce to 8 if you get CUDA OOM
    show_progress_bar=True,
)

dense_vecs    = encoded["dense_vecs"]       # shape: (n_chunks, 1024)
sparse_weights = encoded["lexical_weights"]  # list of dicts

print(f"Dense shape: {dense_vecs.shape}")
print(f"Sparse entries (sample): {list(sparse_weights[0].items())[:5]}")

In [ ]:
import chromadb
import json
import numpy as np

# Persistent storage - survives kernel restarts
client = chromadb.PersistentClient(path="./chroma_db")

# Delete collection if rebuilding from scratch
try:
    client.delete_collection("redteam_rag")
except:
    pass

collection = client.create_collection(
    name="redteam_rag",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for dense retrieval
)

# ChromaDB requires list[str] for ids
ids       = [str(i) for i in range(len(all_chunks))]
documents = [c["text"] for c in all_chunks]

# Metadata must be flat (no nested lists) - serialise lists to strings
metadatas = []
for c in all_chunks:
    meta = dict(c["metadata"])
    # Flatten any list fields
    for k, v in meta.items():
        if isinstance(v, list):
            meta[k] = ", ".join(v) if v else ""
    metadatas.append(meta)

# Add in batches to avoid memory issues
BATCH_SIZE = 500
for i in range(0, len(all_chunks), BATCH_SIZE):
    end = min(i + BATCH_SIZE, len(all_chunks))
    collection.add(
        ids=ids[i:end],
        documents=documents[i:end],
        embeddings=dense_vecs[i:end].tolist(),
        metadatas=metadatas[i:end],
    )
    print(f"Stored batch {i}:{end}")

print(f"Total stored in ChromaDB: {collection.count()}")

In [ ]:
import pickle

sparse_store = {
    str(i): sparse_weights[i] 
    for i in range(len(all_chunks))
}

with open("sparse_weights.pkl", "wb") as f:
    pickle.dump(sparse_store, f)

print("Sparse weights saved")

In [ ]:
# Quick retrieval test
test_query = "How do I perform lateral movement using Pass-the-Hash?"

# Embed the query
q_encoded = embedder.encode(
    [test_query],
    return_dense=True,
    return_sparse=True,
)

# Dense retrieval from ChromaDB
results = collection.query(
    query_embeddings=q_encoded["dense_vecs"].tolist(),
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

print("Top 5 dense retrieval results:\n")
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    print(f"Rank {i+1} | Score: {1-dist:.3f} | Source: {meta['source']} | {meta.get('id', meta.get('section', ''))}")
    print(f"  {doc[:150]}...")
    print()

In [ ]:
print(f"ChromaDB collection count: {collection.count()}")
print(f"Sparse weights stored: {len(sparse_store)}")
assert collection.count() == len(sparse_store) == len(all_chunks), \
    "Mismatch between stores — recheck ingestion"
print("All stores consistent ✓")